# Step 3: Probe Training & Analysis

In this notebook we train **linear probes** on the hidden-state activations extracted in Step 2 to determine whether and *where* transformer layers encode corporate identity information.

The core idea follows the linearity hypothesis (Goldowsky-Dill et al., Neel Nanda et al.): if a concept is represented as a linear direction in activation space, a simple logistic-regression probe trained on those activations should achieve above-chance classification.

**Outline:**
1. Load saved activations
2. Train a binary probe (Anthropic vs OpenAI)
3. Sweep across all layers to find peak encoding depth
4. Train a multi-class probe across all identities
5. Compare against random and surface baselines
6. Control for content by restricting to neutral queries
7. Visualize identity clusters via PCA
8. Compare peak layers with evaluation-awareness literature

In [ ]:
import sys
import numpy as np
import torch
import pickle
from pathlib import Path

# Ensure the project root is on the path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from research.config import (
    ACTIVATIONS_DIR, PROBES_DIR, FIGURES_DIR,
    model_config, experiment_config, IDENTITY_CONDITIONS,
)
from research.probing.linear_probe import CorporateIdentityProbe
from research.probing.analysis import ProbeAnalyzer
from research.utils.visualization import ResearchVisualizer

print(f"Activations dir : {ACTIVATIONS_DIR}")
print(f"Probes dir      : {PROBES_DIR}")
print(f"Figures dir     : {FIGURES_DIR}")
print(f"Model           : {model_config.model_name}")
print(f"Num layers      : {model_config.num_layers}")
print(f"Hidden dim      : {model_config.hidden_dim}")

## 3.1 Loading Activations

In the previous notebook we extracted hidden-state activations for every (identity, query) pair and saved them to disk. Each activation tensor has shape `(num_layers, hidden_dim)`, capturing the residual stream at the last token position across all transformer layers.

We load these into a nested dictionary: `activations[identity][query] -> tensor`.

In [ ]:
# Load saved activations from outputs/activations/
activations_path = ACTIVATIONS_DIR / "all_activations.pkl"
with open(activations_path, "rb") as f:
    activations = pickle.load(f)

# Inspect the structure
print("Identities loaded:", list(activations.keys()))
for identity in activations:
    queries = list(activations[identity].keys())
    sample_tensor = activations[identity][queries[0]]
    print(f"  {identity}: {len(queries)} queries, tensor shape = {sample_tensor.shape}")

## 3.2 Binary Probe: Anthropic vs OpenAI

We start with the simplest test: can a linear classifier distinguish Anthropic-prompted activations from OpenAI-prompted activations at a single layer?

This binary probe approach follows Goldowsky-Dill et al.'s methodology for detecting linearly-encoded concepts in transformer residual streams. If the probe achieves significantly above-chance AUROC, the identity information is linearly accessible at that layer.

We pick the middle layer as a reasonable first guess, then do a full sweep in section 3.3.

In [ ]:
probe = CorporateIdentityProbe()

# Prepare data at a single layer (middle of the network)
test_layer = model_config.num_layers // 2
print(f"Training binary probe at layer {test_layer}...")

# Extract positive (anthropic) and negative (openai) activations
X_anthropic = []
for query, tensor in activations["anthropic"].items():
    if hasattr(tensor, "cpu"):
        X_anthropic.append(tensor[test_layer].cpu().numpy())
    else:
        X_anthropic.append(np.asarray(tensor[test_layer]))
X_anthropic = np.stack(X_anthropic)

X_openai = []
for query, tensor in activations["openai"].items():
    if hasattr(tensor, "cpu"):
        X_openai.append(tensor[test_layer].cpu().numpy())
    else:
        X_openai.append(np.asarray(tensor[test_layer]))
X_openai = np.stack(X_openai)

# Train the binary probe
binary_result = probe.train_binary_probe(X_anthropic, X_openai)

print(f"\n--- Binary Probe Results (Layer {test_layer}) ---")
print(f"  AUROC    : {binary_result['auroc']:.4f}")
print(f"  Accuracy : {binary_result['accuracy']:.4f}")
print(f"  F1       : {binary_result['f1']:.4f}")
print(f"  CV scores: {[f'{s:.3f}' for s in binary_result['cv_scores']]}")
print(f"  CV mean  : {np.mean(binary_result['cv_scores']):.4f} +/- {np.std(binary_result['cv_scores']):.4f}")

## 3.3 Layer Sweep

A single-layer probe tells us *whether* identity is detectable, but not *where* the representation is strongest. Different types of information are encoded at different depths:

- **Early layers** tend to encode syntactic and positional features
- **Middle layers** often capture semantic content
- **Late layers** specialize in task-specific and output-oriented representations

We now train a binary probe at every layer to build an accuracy-by-layer profile. This reveals the layer(s) where corporate identity is most linearly separable.

In [ ]:
# Run layer sweep: binary probe Anthropic vs OpenAI at every layer
print(f"Running layer sweep across {model_config.num_layers} layers...")
print("(This may take a few minutes)\n")

sweep_results = probe.layer_sweep(
    activations,
    probe_type="binary",
    identity_pair=("anthropic", "openai"),
)

# Print per-layer results
print(f"{'Layer':>5}  {'AUROC':>7}  {'Accuracy':>8}  {'F1':>7}  {'CV Mean':>8}")
print("-" * 45)
for layer_idx in sorted(sweep_results.keys()):
    r = sweep_results[layer_idx]
    cv_mean = np.mean(r['cv_scores'])
    print(f"{layer_idx:>5}  {r['auroc']:>7.4f}  {r['accuracy']:>8.4f}  {r['f1']:>7.4f}  {cv_mean:>8.4f}")

## 3.4 Visualizing Layer Accuracy

We use the `ProbeAnalyzer` to plot the AUROC curve across layers and identify the peak layers where corporate identity is most strongly encoded.

In [ ]:
# Wrap results for the analyzer
analyzer = ProbeAnalyzer({"anthropic_vs_openai": sweep_results})

# Plot layer accuracy curve
fig = analyzer.plot_layer_accuracy(
    save_path=FIGURES_DIR / "probe_layer_accuracy.png"
)
fig  # display in notebook

# Find peak layers
peak_layers = analyzer.find_peak_layers(
    metric="auroc", top_k=5, sweep_key="anthropic_vs_openai"
)

print("\nTop 5 layers by AUROC:")
for layer_idx, auroc_val in peak_layers:
    print(f"  Layer {layer_idx}: AUROC = {auroc_val:.4f}")

# Store the best layer for downstream use
peak_layer = peak_layers[0][0]
print(f"\nPeak layer selected: {peak_layer}")

## 3.5 Multi-class Probe

The binary probe tells us that two specific identities are distinguishable. But can a single probe classify *all* identity conditions simultaneously?

We now train a **one-vs-rest multiclass logistic regression** at the peak layer, classifying among all identities (Anthropic, OpenAI, Google, Meta, neutral, none). The confusion matrix reveals which identities are most confusable, hinting at the structure of the identity representation.

In [ ]:
# Prepare multiclass data at the peak layer
X_multi, y_multi, label_enc = probe.prepare_data(activations, layer=peak_layer)

print(f"Multiclass probe at layer {peak_layer}")
print(f"  Samples: {X_multi.shape[0]}, Features: {X_multi.shape[1]}")
print(f"  Classes: {list(label_enc.classes_)}")

# Train multiclass probe
multi_result = probe.train_multiclass_probe(X_multi, y_multi)

print(f"\n--- Multiclass Probe Results ---")
print(f"  Accuracy  : {multi_result['accuracy']:.4f}")
print(f"  F1 (macro): {multi_result['f1_macro']:.4f}")
print(f"  CV scores : {[f'{s:.3f}' for s in multi_result['cv_scores']]}")
print(f"  CV mean   : {np.mean(multi_result['cv_scores']):.4f} +/- {np.std(multi_result['cv_scores']):.4f}")

# Plot confusion matrix
fig_cm = analyzer.plot_confusion_matrix(
    cm=multi_result["confusion_matrix"],
    labels=list(label_enc.classes_),
    save_path=FIGURES_DIR / "probe_confusion_matrix.png",
)
fig_cm

## 3.6 Baselines

A high probe accuracy is only meaningful relative to appropriate baselines. We compare against two controls:

1. **Random baseline**: project activations onto a random direction and threshold. This estimates chance-level performance and ensures our probe is learning a real signal rather than exploiting geometric artifacts.

2. **Surface baseline**: train a logistic regression on raw bag-of-tokens features from the input. If the surface baseline matches the activation-based probe, the probe may be detecting input artifacts rather than learned internal representations.

In [ ]:
# 1. Random baseline
X_binary = np.concatenate([X_anthropic, X_openai], axis=0)
y_binary = np.concatenate([
    np.ones(len(X_anthropic), dtype=int),
    np.zeros(len(X_openai), dtype=int),
])

random_baseline = probe.train_random_baseline(X_binary, y_binary)

print("--- Random Baseline ---")
print(f"  Accuracy : {random_baseline['accuracy']:.4f}")
print(f"  F1       : {random_baseline['f1']:.4f}")
print(f"  AUROC    : {random_baseline['auroc']:.4f}")

# 2. Surface baseline (using tokenized inputs if available)
# For the surface baseline, we need the raw tokenized prompts.
# Load them from the saved dataset or reconstruct from identities/queries.
surface_tokens_path = ACTIVATIONS_DIR / "tokenized_inputs.pkl"
try:
    with open(surface_tokens_path, "rb") as f:
        tokenized_inputs_data = pickle.load(f)
    
    # Build parallel lists matching the binary probe order
    surface_tokens = (
        tokenized_inputs_data["anthropic"] + tokenized_inputs_data["openai"]
    )
    surface_baseline = probe.train_surface_baseline(surface_tokens, y_binary)
    
    print("\n--- Surface Baseline ---")
    print(f"  Accuracy : {surface_baseline['accuracy']:.4f}")
    print(f"  F1       : {surface_baseline['f1']:.4f}")
    print(f"  CV scores: {[f'{s:.3f}' for s in surface_baseline['cv_scores']]}")
except FileNotFoundError:
    print("\n[INFO] Tokenized inputs not found; skipping surface baseline.")
    print("       Re-run Step 2 with save_tokenized_inputs=True to enable this.")
    surface_baseline = None

# Comparison summary
print("\n=== Baseline Comparison ===")
print(f"  Real probe (layer {peak_layer}) AUROC : {sweep_results[peak_layer]['auroc']:.4f}")
print(f"  Random baseline AUROC               : {random_baseline['auroc']:.4f}")
if surface_baseline:
    print(f"  Surface baseline accuracy            : {surface_baseline['accuracy']:.4f}")

## 3.7 Content Controls

One concern is that the probe might be detecting *query content* rather than *identity encoding*. For instance, if certain queries are only asked under certain identity conditions, the probe could learn to classify queries rather than identity.

To control for this, we filter the activations to include only **neutral queries** (queries that are not specifically about AI companies or identities) and retrain the probe. If accuracy holds, the probe is genuinely detecting identity representations rather than content artifacts.

In [ ]:
# Define neutral queries (those without identity-specific content)
# We filter by checking whether the query text contains company names
company_terms = [
    "anthropic", "claude", "openai", "chatgpt", "gpt",
    "google", "deepmind", "gemini", "meta", "llama", "facebook",
]

def is_neutral_query(query_text: str) -> bool:
    lower = query_text.lower()
    return not any(term in lower for term in company_terms)

# Build filtered activations dict
filtered_activations = {}
for identity, queries in activations.items():
    filtered = {q: t for q, t in queries.items() if is_neutral_query(q)}
    if filtered:
        filtered_activations[identity] = filtered

print("Filtered to neutral queries only:")
for identity in filtered_activations:
    print(f"  {identity}: {len(filtered_activations[identity])} queries")

# Retrain binary probe at peak layer on neutral queries
if "anthropic" in filtered_activations and "openai" in filtered_activations:
    X_anthro_filt = []
    for q, t in filtered_activations["anthropic"].items():
        vec = t[peak_layer].cpu().numpy() if hasattr(t, "cpu") else np.asarray(t[peak_layer])
        X_anthro_filt.append(vec)
    X_anthro_filt = np.stack(X_anthro_filt)

    X_openai_filt = []
    for q, t in filtered_activations["openai"].items():
        vec = t[peak_layer].cpu().numpy() if hasattr(t, "cpu") else np.asarray(t[peak_layer])
        X_openai_filt.append(vec)
    X_openai_filt = np.stack(X_openai_filt)

    neutral_result = probe.train_binary_probe(X_anthro_filt, X_openai_filt)

    print(f"\n--- Neutral-Query Binary Probe (Layer {peak_layer}) ---")
    print(f"  AUROC    : {neutral_result['auroc']:.4f}")
    print(f"  Accuracy : {neutral_result['accuracy']:.4f}")
    print(f"  F1       : {neutral_result['f1']:.4f}")
    print(f"\n  Compared to full-data AUROC: {sweep_results[peak_layer]['auroc']:.4f}")
    delta = neutral_result['auroc'] - sweep_results[peak_layer]['auroc']
    print(f"  Delta: {delta:+.4f}")
else:
    print("\n[WARNING] Insufficient neutral queries for both identities.")

## 3.8 PCA Visualization

While probes quantify linear separability, PCA gives us an intuitive view of the geometry. We project the high-dimensional activations at the peak layer down to 2D and color each point by its identity condition.

Clear clustering suggests identity is a dominant source of variance at this layer; overlapping clusters suggest the representation is more distributed or entangled with other features.

In [ ]:
# PCA of activations at the peak layer colored by identity
fig_pca = analyzer.plot_probe_direction_pca(
    activations=activations,
    layer=peak_layer,
    save_path=FIGURES_DIR / "pca_identity_clusters.png",
)
fig_pca

## 3.9 Comparison with Evaluation Awareness

Nguyen et al. (2025) found that evaluation-awareness representations in Llama-2-70B concentrate in **layers 23-24**. If corporate identity encoding peaks at similar depths, this would suggest the two phenomena share a common representational substrate -- potentially a general "situational awareness" subspace.

If the peaks are distant, identity and eval-awareness may be encoded independently, which has different implications for targeted interventions.

In [ ]:
# Compare our peak layers with eval-awareness literature
corporate_peaks = [layer_idx for layer_idx, _ in peak_layers[:3]]

comparison = analyzer.compare_with_eval_awareness(
    corporate_peak_layers=corporate_peaks,
    eval_awareness_layers=[23, 24],  # Nguyen et al. (2025)
)

print("=== Comparison with Evaluation Awareness ===")
print(f"  Corporate identity peaks : {comparison['corporate_peaks']}")
print(f"  Eval-awareness layers    : {comparison['eval_awareness_layers']}")
print(f"  Overlapping layers       : {comparison['overlap']}")
print(f"  Mean layer distance      : {comparison['mean_distance']:.1f}")
print(f"\n  Interpretation: {comparison['interpretation']}")

## 3.10 Summary

We now generate a comprehensive report summarizing all probe findings. Key questions answered:

- **Is corporate identity linearly encoded?** (Binary probe AUROC >> 0.5)
- **Where is it strongest?** (Peak layer from layer sweep)
- **Is it multi-class separable?** (Multiclass accuracy and confusion patterns)
- **Is it a real signal?** (Comparison with random and surface baselines)
- **Is it content-independent?** (Neutral-query control)
- **Does it overlap with eval-awareness?** (Layer comparison with Nguyen et al.)

In [ ]:
# Generate the full probe analysis report
report = analyzer.generate_report()
print(report)

# Save probe results for use in Notebook 04 (steering)
probe_output = {
    "peak_layer": peak_layer,
    "peak_layers_top5": peak_layers,
    "binary_sweep_results": sweep_results,
    "multiclass_result": {
        "accuracy": multi_result["accuracy"],
        "f1_macro": multi_result["f1_macro"],
        "cv_scores": multi_result["cv_scores"],
        "confusion_matrix": multi_result["confusion_matrix"],
        "classes": list(label_enc.classes_),
    },
    "direction_vector": sweep_results[peak_layer]["direction"],
    "comparison_with_eval_awareness": comparison,
}

output_path = PROBES_DIR / "probe_results.pkl"
with open(output_path, "wb") as f:
    pickle.dump(probe_output, f)
print(f"\nProbe results saved to: {output_path}")